# Modulo 10 – Kubernetes e TensorFlow Serving (ML Zoomcamp)

Appunti sulle nozioni fondamentali del modulo [10-kubernetes](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/main/10-kubernetes) di DataTalksClub.

**Contenuti**
1. Panoramica e architettura
2. TensorFlow Serving
3. Gateway (pre-processing) e architettura a due servizi
4. Concetti base di Kubernetes (Node, Pod, Deployment, Service)
5. Cluster locale con `kind` e comandi `kubectl`
6. Deployment e Service di un servizio semplice
7. Deployment di TF-Serving + Gateway su Kubernetes
8. Deploy su AWS EKS
9. Autoscaling (HPA)
10. Cheat sheet e errori comuni

> Nota: le celle di codice contengono comandi shell (`%%bash`) e file YAML da eseguire/salvare in locale. Richiedono Docker, `kind` e `kubectl` installati.

## 1. Panoramica e architettura

Obiettivo del modulo: portare in produzione un modello di deep learning (classificazione di vestiti, rete **Xception**) usando **Kubernetes**.

L'architettura è divisa in due componenti:

```
Client ──► Gateway (Flask, :9696) ──► TF-Serving (modello, :8500) 
            │ scarica immagine         │ esegue solo l'inferenza
            │ pre-processing           │ (gRPC)
            │ post-processing
```

| Componente | Compito | Risorse |
|---|---|---|
| **TF-Serving** | Serve il modello in formato SavedModel via gRPC | CPU/RAM pesanti (modello) |
| **Gateway** | Riceve URL immagine, fa pre-processing, chiama TF-Serving, restituisce le label | Leggero (CPU) |

**Perché separarli?** Si possono scalare in modo indipendente e usare risorse diverse (es. il modello su macchine con GPU, il gateway su macchine economiche).

## 2. TensorFlow Serving

TF-Serving è un sistema ad alte prestazioni per servire modelli TensorFlow.

Passi:
1. Convertire il modello Keras (`.h5`) nel formato **SavedModel**.
2. Ispezionare input/output con `saved_model_cli`.
3. Avviare TF-Serving con Docker.

In [ ]:
%%bash
# 1) Conversione Keras -> SavedModel (eseguire in Python)
python - << 'EOF'
import tensorflow as tf
from tensorflow import keras
model = keras.models.load_model('xception_v4_large_08_0.894.h5')
tf.saved_model.save(model, 'clothing-model')
EOF

# 2) Ispezione della signature (nome input/output)
saved_model_cli show --dir clothing-model --all

In [ ]:
%%bash
# 3) Avvio di TF-Serving in locale
docker run -it --rm \
  -p 8500:8500 \
  -v "$(pwd)/clothing-model:/models/clothing-model/1" \
  -e MODEL_NAME="clothing-model" \
  tensorflow/serving:2.7.0

In [ ]:
# Client gRPC minimo (esempio concettuale)
import grpc
import tensorflow as tf
from tensorflow_serving.apis import predict_pb2, prediction_service_pb2_grpc
from keras_image_helper import create_preprocessor

host = 'localhost:8500'
channel = grpc.insecure_channel(host)
stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)

preprocessor = create_preprocessor('xception', target_size=(299, 299))

def np_to_protobuf(data):
    return tf.make_tensor_proto(data, shape=data.shape)

def prepare_request(X):
    pb_request = predict_pb2.PredictRequest()
    pb_request.model_spec.name = 'clothing-model'
    pb_request.model_spec.signature_name = 'serving_default'
    pb_request.inputs['input_8'].CopyFrom(np_to_protobuf(X))   # il nome dipende dal modello
    return pb_request

classes = ['dress','hat','longsleeve','outwear','pants','shirt','shoes','shorts','skirt','t-shirt']

def predict(url):
    X = preprocessor.from_url(url)
    pb_request = prepare_request(X)
    pb_response = stub.Predict(pb_request, timeout=20.0)
    preds = pb_response.outputs['dense_7'].float_val   # il nome dipende dal modello
    return dict(zip(classes, preds))

## 3. Gateway Flask

Il gateway incapsula pre-processing e post-processing e comunica con TF-Serving.
L'indirizzo di TF-Serving è letto da una **variabile d'ambiente**, così lo stesso container funziona in locale (`localhost:8500`) e in Kubernetes (nome del Service, es. `tf-serving-clothing-model.default.svc.cluster.local:8500` oppure `tf-serving-clothing-model:8500`).

In [ ]:
# gateway.py
import os
from flask import Flask, request, jsonify

app = Flask('gateway')
host = os.getenv('TF_SERVING_HOST', 'localhost:8500')

# ... (channel/stub/preprocessor come sopra, con 'host')

@app.route('/predict', methods=['POST'])
def predict_endpoint():
    data = request.get_json()
    return jsonify(predict(data['url']))

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=9696)

In [ ]:
%%bash
# Dockerfile di TF-Serving (modello incluso nell'immagine)
cat > image-model.dockerfile << 'EOF'
FROM tensorflow/serving:2.7.0
COPY clothing-model /models/clothing-model/1
ENV MODEL_NAME="clothing-model"
EOF

# Dockerfile del gateway
cat > image-gateway.dockerfile << 'EOF'
FROM python:3.8.12-slim
RUN pip install pipenv
WORKDIR /app
COPY ["Pipfile", "Pipfile.lock", "./"]
RUN pipenv install --system --deploy
COPY ["gateway.py", "proto.py", "./"]
EXPOSE 9696
ENTRYPOINT ["gunicorn", "--bind=0.0.0.0:9696", "gateway:app"]
EOF

docker build -t zoomcamp-10-model:xception-v4-001 -f image-model.dockerfile .
docker build -t zoomcamp-10-gateway:002 -f image-gateway.dockerfile .

In [ ]:
%%bash
# Test locale con docker-compose (alternativa: rete docker manuale)
cat > docker-compose.yaml << 'EOF'
services:
  clothing-model:
    image: zoomcamp-10-model:xception-v4-001
  gateway:
    image: zoomcamp-10-gateway:002
    environment:
      - TF_SERVING_HOST=clothing-model:8500
    ports:
      - "9696:9696"
EOF
docker compose up

## 4. Concetti fondamentali di Kubernetes

**Kubernetes (K8s)** è un sistema open source per **orchestrare** container: deployment, scaling, rete e self-healing.

### Architettura del cluster

| Elemento | Descrizione |
|---|---|
| **Cluster** | Insieme di macchine (nodi) gestite da K8s |
| **Control plane** | Cervello del cluster: API server, scheduler, controller manager, etcd |
| **Node** | Macchina (fisica/VM) che esegue i workload; contiene kubelet e container runtime |
| **Pod** | Unità minima schedulabile: uno o più container che condividono rete e storage. **Efimero** |
| **Deployment** | Descrive lo stato desiderato di un insieme di Pod (immagine, repliche, risorse); gestisce rollout e self-healing tramite ReplicaSet |
| **Service** | IP/nome DNS **stabile** che bilancia il traffico verso un gruppo di Pod (selezionati tramite **label**) |
| **Ingress** | Espone servizi HTTP/HTTPS all'esterno con regole di routing |
| **HPA** | Horizontal Pod Autoscaler: modifica il numero di repliche in base a metriche (CPU, ecc.) |

### Tipi di Service

| Tipo | Accessibilità | Uso tipico |
|---|---|---|
| **ClusterIP** (default) | Solo interna al cluster | Il modello TF-Serving (non deve essere esposto) |
| **NodePort** | Porta su ogni nodo (30000-32767) | Test, sviluppo |
| **LoadBalancer** | IP esterno tramite load balancer del cloud | Il gateway in produzione (es. EKS) |
| **ExternalName** | Alias DNS verso un servizio esterno | Raro |

Nel modulo: **TF-Serving → ClusterIP** (interno), **Gateway → LoadBalancer** (esterno).

### Dichiarativo
Si descrive lo stato desiderato in file **YAML** (manifest) e si applica con `kubectl apply -f`. Kubernetes lavora continuamente per far coincidere lo stato reale con quello desiderato.

## 5. Cluster locale con `kind` e comandi `kubectl`

**kind** (Kubernetes IN Docker) crea cluster K8s locali usando container Docker come nodi. **kubectl** è la CLI per interagire con il cluster.

In [ ]:
%%bash
# Installazione (esempio Linux; su macOS: brew install kind kubectl)
# kind:    https://kind.sigs.k8s.io/docs/user/quick-start/
# kubectl: https://kubernetes.io/docs/tasks/tools/

kind --version
kubectl version --client

In [ ]:
%%bash
# Creazione del cluster
kind create cluster                      # nome di default: kind
# oppure, con un nome e/o un file di configurazione:
# kind create cluster --name ml-zoomcamp --config kind.yaml

# Verifiche
kubectl cluster-info --context kind-kind
kubectl get nodes
kubectl get service                      # c'è già il service 'kubernetes'

In [ ]:
%%bash
# Esempio di kind.yaml: 1 control plane + 2 worker
cat > kind.yaml << 'EOF'
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
  - role: control-plane
  - role: worker
  - role: worker
EOF

In [ ]:
%%bash
# Caricare un'immagine Docker locale nel cluster kind
# (kind NON vede automaticamente le immagini del Docker locale!)
kind load docker-image zoomcamp-10-model:xception-v4-001
kind load docker-image zoomcamp-10-gateway:002

# Eliminare il cluster
# kind delete cluster

### Comandi `kubectl` principali

| Comando | Funzione |
|---|---|
| `kubectl get nodes / pods / deployments / services` | Elenca risorse |
| `kubectl get pod -o wide` | Dettagli extra (IP, nodo) |
| `kubectl describe pod <nome>` | Dettaglio ed eventi (utile per il debug) |
| `kubectl logs <pod>` | Log del container |
| `kubectl apply -f file.yaml` | Crea/aggiorna risorse da manifest |
| `kubectl delete -f file.yaml` / `kubectl delete pod <nome>` | Elimina risorse |
| `kubectl port-forward <pod\|service/nome> 8080:80` | Inoltra una porta locale verso il cluster |
| `kubectl exec -it <pod> -- bash` | Shell dentro un container |
| `kubectl scale deployment <nome> --replicas=3` | Cambia le repliche |
| `kubectl rollout status deployment/<nome>` | Stato di un aggiornamento |
| `kubectl config get-contexts` / `use-context <ctx>` | Gestione contesti (cluster) |

## 6. Deployment e Service di un servizio semplice

Esempio: un servizio di predizione (es. churn/subscription) in un container che espone la porta **9696**.

### Deployment (`deployment.yaml`)
Punti chiave:
- `replicas`: numero di Pod
- `selector.matchLabels` deve coincidere con `template.metadata.labels`
- `resources.limits`: memoria e CPU massime (`cpu: "100m"` = 0,1 core)
- `containerPort`: porta esposta dall'immagine

In [ ]:
%%bash
cat > deployment.yaml << 'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: subscription
spec:
  replicas: 1
  selector:
    matchLabels:
      app: subscription
  template:
    metadata:
      labels:
        app: subscription
    spec:
      containers:
      - name: subscription
        image: zoomcamp-model:3.13.10-hw10
        imagePullPolicy: IfNotPresent     # necessario con immagini caricate via 'kind load'
        resources:
          requests:
            memory: "64Mi"
            cpu: "100m"
          limits:
            memory: "128Mi"
            cpu: "200m"
        ports:
        - containerPort: 9696
EOF

### Service (`service.yaml`)
- `selector` seleziona i Pod tramite label (`app: subscription`)
- `port`: porta del Service; `targetPort`: porta del container
- Senza `type`, il default è **ClusterIP**

In [ ]:
%%bash
cat > service.yaml << 'EOF'
apiVersion: v1
kind: Service
metadata:
  name: subscription
spec:
  type: ClusterIP
  selector:
    app: subscription
  ports:
  - port: 80
    targetPort: 9696
EOF

kubectl apply -f deployment.yaml
kubectl apply -f service.yaml

kubectl get pods
kubectl get service

# Test: inoltro della porta (in un altro terminale) e richiesta
# kubectl port-forward service/subscription 8080:80
# curl -X POST localhost:8080/predict -H 'Content-Type: application/json' -d '{"...": "..."}'

## 7. Deployment di TF-Serving + Gateway su Kubernetes

Sequenza consigliata (verificare ogni passo con `port-forward`):

1. **Model deployment** → test del Pod
2. **Model service** (ClusterIP, porta 8500) → test del service
3. **Gateway deployment** (con env `TF_SERVING_HOST`) → test del Pod
4. **Gateway service** (LoadBalancer) → test finale

In [ ]:
%%bash
cat > model-deployment.yaml << 'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: tf-serving-clothing-model
spec:
  replicas: 1
  selector:
    matchLabels:
      app: tf-serving-clothing-model
  template:
    metadata:
      labels:
        app: tf-serving-clothing-model
    spec:
      containers:
      - name: tf-serving-clothing-model
        image: zoomcamp-10-model:xception-v4-001
        resources:
          limits:
            memory: "512Mi"
            cpu: "1"
        ports:
        - containerPort: 8500
EOF

cat > model-service.yaml << 'EOF'
apiVersion: v1
kind: Service
metadata:
  name: tf-serving-clothing-model
spec:
  type: ClusterIP            # default: solo interno
  selector:
    app: tf-serving-clothing-model
  ports:
  - port: 8500
    targetPort: 8500
EOF

kubectl apply -f model-deployment.yaml
kubectl apply -f model-service.yaml
kubectl get pod
kubectl get service
# kubectl port-forward service/tf-serving-clothing-model 8500:8500   # poi esegui il client gRPC

In [ ]:
%%bash
cat > gateway-deployment.yaml << 'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: gateway
spec:
  replicas: 1
  selector:
    matchLabels:
      app: gateway
  template:
    metadata:
      labels:
        app: gateway
    spec:
      containers:
      - name: gateway
        image: zoomcamp-10-gateway:002
        resources:
          limits:
            memory: "128Mi"
            cpu: "100m"
        ports:
        - containerPort: 9696
        env:
          - name: TF_SERVING_HOST
            value: "tf-serving-clothing-model.default.svc.cluster.local:8500"
EOF

cat > gateway-service.yaml << 'EOF'
apiVersion: v1
kind: Service
metadata:
  name: gateway
spec:
  type: LoadBalancer         # esposto all'esterno (su kind: serve port-forward o MetalLB)
  selector:
    app: gateway
  ports:
  - port: 80
    targetPort: 9696
EOF

kubectl apply -f gateway-deployment.yaml
kubectl apply -f gateway-service.yaml
kubectl get service

# In locale con kind:
# kubectl port-forward service/gateway 8080:80

In [ ]:
# test.py – richiamo del gateway
import requests

url = 'http://localhost:8080/predict'
data = {'url': 'http://bit.ly/mlbookcamp-pants'}
print(requests.post(url, json=data).json())

**DNS interno:** i Service sono raggiungibili con il nome DNS `<service>.<namespace>.svc.cluster.local` (es. `tf-serving-clothing-model.default.svc.cluster.local`). Nello stesso namespace basta anche solo `<service>`.

## 8. Deploy su AWS EKS

Elastic Kubernetes Service è il Kubernetes gestito di AWS. Si usa **`eksctl`** per creare il cluster e **ECR** come registry per le immagini.

Passi:
1. Installare e configurare `aws cli`, `eksctl`, `kubectl`
2. Creare un cluster con un file di configurazione
3. Creare un repository ECR e fare il push delle immagini
4. Aggiornare i manifest con gli URI delle immagini su ECR
5. `kubectl apply` e ottenere l'**EXTERNAL-IP** del LoadBalancer

In [ ]:
%%bash
# eks-config.yaml
cat > eks-config.yaml << 'EOF'
apiVersion: eksctl.io/v1alpha5
kind: ClusterConfig
metadata:
  name: mlzoomcamp-eks
  region: eu-west-1
nodeGroups:
  - name: ng-m5-xlarge
    instanceType: m5.xlarge
    desiredCapacity: 1
EOF

eksctl create cluster -f eks-config.yaml

In [ ]:
%%bash
# Registry ECR
aws ecr create-repository --repository-name mlzoomcamp-images

ACCOUNT_ID=<IL_TUO_ACCOUNT_ID>
REGION=eu-west-1
REGISTRY_NAME=mlzoomcamp-images
PREFIX=${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com/${REGISTRY_NAME}

# Login a ECR
aws ecr get-login-password --region ${REGION} | \
  docker login --username AWS --password-stdin ${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com

# Tag e push (nota: ':' del tag locale diventa '-' nel tag remoto)
GATEWAY_LOCAL=zoomcamp-10-gateway:002
GATEWAY_REMOTE=${PREFIX}:zoomcamp-10-gateway-002
docker tag ${GATEWAY_LOCAL} ${GATEWAY_REMOTE}
docker push ${GATEWAY_REMOTE}

MODEL_LOCAL=zoomcamp-10-model:xception-v4-001
MODEL_REMOTE=${PREFIX}:zoomcamp-10-model-xception-v4-001
docker tag ${MODEL_LOCAL} ${MODEL_REMOTE}
docker push ${MODEL_REMOTE}

In [ ]:
%%bash
# Dopo aver sostituito le 'image:' nei manifest con gli URI ECR
kubectl get nodes
kubectl apply -f model-deployment.yaml
kubectl apply -f model-service.yaml
kubectl apply -f gateway-deployment.yaml
kubectl apply -f gateway-service.yaml

kubectl get service     # copia EXTERNAL-IP del gateway e usalo in test.py

# IMPORTANTE: eliminare il cluster per non pagare!
# eksctl delete cluster --name mlzoomcamp-eks

## 9. Autoscaling con HPA (Horizontal Pod Autoscaler)

L'HPA aumenta/riduce il numero di Pod in base all'utilizzo (tipicamente CPU).

Requisiti:
- Il **metrics-server** deve essere installato (su kind va installato a mano, con l'opzione `--kubelet-insecure-tls`)
- Il Deployment deve definire `resources.requests.cpu`, altrimenti la percentuale è `<unknown>`

In [ ]:
%%bash
# Metrics server
kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
# Su kind: patch per accettare i certificati kubelet non verificati
kubectl patch -n kube-system deployment metrics-server --type=json \
  -p '[{"op":"add","path":"/spec/template/spec/containers/0/args/-","value":"--kubelet-insecure-tls"}]'

# Creazione dell'HPA: 1-3 repliche, target CPU 20%
kubectl autoscale deployment subscription --name=subscription-hpa --cpu-percent=20 --min=1 --max=3

kubectl get hpa
# NAME               REFERENCE                 TARGETS   MINPODS   MAXPODS   REPLICAS
# subscription-hpa   Deployment/subscription   1%/20%    1         3         1

In [ ]:
%%bash
# Equivalente dichiarativo
cat > hpa.yaml << 'EOF'
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: subscription-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: subscription
  minReplicas: 1
  maxReplicas: 3
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 20
EOF
kubectl apply -f hpa.yaml

# Generare carico e osservare lo scaling
# kubectl get hpa subscription-hpa --watch

## 10. Cheat sheet e errori comuni

### Flusso tipico (kind)
```
docker build  →  kind create cluster  →  kind load docker-image
      →  kubectl apply -f deployment.yaml / service.yaml
      →  kubectl get pods/svc  →  kubectl port-forward  →  test
```

### Errori frequenti
| Sintomo | Causa probabile | Soluzione |
|---|---|---|
| `ErrImagePull` / `ImagePullBackOff` | Immagine non caricata in kind o tag `latest` | `kind load docker-image ...` e `imagePullPolicy: IfNotPresent`, usare un tag esplicito |
| `CrashLoopBackOff` | Errore nell'app / variabile d'ambiente mancante | `kubectl logs <pod>`, `kubectl describe pod <pod>` |
| `OOMKilled` | Limite di memoria troppo basso | Aumentare `resources.limits.memory` |
| Service senza endpoint | `selector` ≠ label dei Pod | Confrontare label con `kubectl get pods --show-labels` |
| `EXTERNAL-IP <pending>` su kind | kind non ha un load balancer cloud | Usare `port-forward` o MetalLB |
| HPA mostra `<unknown>` | Manca metrics-server o `requests.cpu` | Installare metrics-server e definire le requests |
| Gateway non raggiunge il modello | Host/porta errati | Usare il nome DNS del Service e la porta 8500 |

### Idee chiave da ricordare
- Un **Pod** è effimero; il **Deployment** lo mantiene vivo e scalabile; il **Service** fornisce un indirizzo stabile.
- Separare modello (TF-Serving) e gateway permette di scalare e dimensionare le risorse separatamente.
- Si usa ClusterIP per i servizi interni e LoadBalancer per quelli esposti.
- `kind load docker-image` è indispensabile per usare immagini locali nel cluster.
- Sul cloud, ricordarsi di **eliminare il cluster** (`eksctl delete cluster`) per evitare costi.

### Risorse
- Repository del modulo: https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/main/10-kubernetes
- Documentazione Kubernetes: https://kubernetes.io/docs/
- kind: https://kind.sigs.k8s.io/
- TensorFlow Serving: https://www.tensorflow.org/tfx/guide/serving